# 03 — Knowledge Extraction

This notebook demonstrates the knowledge extraction pipeline for NeuroForge.
It uses LLM-powered structured output to extract topics, concepts, and relationships
from document chunks.

---

## Topic & Concept Extraction

The `TopicExtractor` class provides the main interface for knowledge extraction:
1. **Topic extraction** — identifies main topics and subtopics from chunk text
2. **Concept extraction** — extracts concepts with definitions, difficulty, prerequisites, and keywords
3. **Batch processing** — processes chunks in groups to stay within LLM context limits
4. **Deduplication** — merges duplicate concepts across chunks
5. **Validation** — ensures JSON output matches Pydantic Concept model

In [ ]:
import sys
sys.path.insert(0, "..")

from models import Chunk, ChunkMetadata, Concept, KnowledgeExtraction
from src.llm import LLMClient
from src.extraction.topics import TopicExtractor

print("Imports OK.")

### Create Sample Chunks

For demonstration, we create sample chunks representing study material:

In [ ]:
sample_chunks = [
    Chunk(
        id="chunk-1",
        content="Machine learning is a branch of artificial intelligence that focuses on "
                "building systems that learn from data. Supervised learning uses labeled "
                "training data to learn a mapping from inputs to outputs.",
        document_id="doc-ml-101",
        chunk_index=0,
        metadata=ChunkMetadata(token_count=40, start_char=0, end_char=200),
    ),
    Chunk(
        id="chunk-2",
        content="Neural networks are computing systems inspired by biological neural networks. "
                "They consist of layers of interconnected nodes (neurons) that process "
                "information. Deep learning uses neural networks with many hidden layers.",
        document_id="doc-ml-101",
        chunk_index=1,
        metadata=ChunkMetadata(token_count=45, start_char=200, end_char=450),
    ),
    Chunk(
        id="chunk-3",
        content="Gradient descent is an optimization algorithm used to minimize the loss "
                "function. It iteratively adjusts model parameters by computing the gradient "
                "and moving in the direction of steepest descent. The learning rate controls "
                "the step size.",
        document_id="doc-ml-101",
        chunk_index=2,
        metadata=ChunkMetadata(token_count=50, start_char=450, end_char=700),
    ),
]

print(f"Created {len(sample_chunks)} sample chunks")
for c in sample_chunks:
    print(f"  [{c.id}] {c.content[:60]}...")

### Initialize the Extractor

The `TopicExtractor` uses the `LLMClient` for all LLM calls.
Make sure your `.env` file has at least one API key configured.

In [ ]:
# Initialize LLM client and extractor
llm_client = LLMClient()
extractor = TopicExtractor(llm_client=llm_client, batch_size=5)

print(f"Available providers: {[p.value for p in llm_client.available_providers]}")
print(f"Batch size: {extractor.batch_size}")

### Step 1: Extract Topics

First, we extract the main topics and subtopics from the chunks:

In [ ]:
if llm_client.available_providers:
    topics = extractor.extract_topics(sample_chunks)
    print(f"Extracted {len(topics)} topics:")
    for t in topics:
        print(f"  - {t}")
else:
    print("No LLM providers available. Set API keys in .env file.")
    print("Using placeholder topics for demonstration.")
    topics = ["Machine Learning", "Neural Networks", "Optimization", "Deep Learning"]
    for t in topics:
        print(f"  - {t}")

### Step 2: Extract Concepts

Using the extracted topics, we now extract concepts with full metadata:

In [ ]:
if llm_client.available_providers:
    concepts = extractor.extract_concepts(sample_chunks, topics)
    print(f"Extracted {len(concepts)} concepts:\n")
    for c in concepts:
        print(f"  [{c.difficulty.value}] {c.name}")
        print(f"    Definition: {c.definition[:80]}...")
        print(f"    Topics: {c.topics}")
        print(f"    Keywords: {c.keywords}")
        print(f"    Prerequisites: {c.prerequisites}")
        print()
else:
    print("No LLM providers available — skipping concept extraction.")
    print("The TopicExtractor.extract_concepts() method would return Concept models")
    print("validated against the Pydantic schema.")

### Step 3: Full Pipeline (extract_batch)

The `extract_batch` method runs the complete pipeline: topics → concepts → relationships

In [ ]:
if llm_client.available_providers:
    extraction = extractor.extract_batch(sample_chunks)
    print(f"Full extraction results:")
    print(f"  Concepts:      {len(extraction.concepts)}")
    print(f"  Relationships: {len(extraction.relationships)}")
    print()
    if extraction.relationships:
        print("Relationships:")
        for r in extraction.relationships:
            print(f"  {r.source_concept} --[{r.relationship_type}]--> {r.target_concept}")
else:
    print("No LLM providers available — skipping full pipeline demo.")
    print("The extract_batch() method returns a KnowledgeExtraction model with:")
    print("  - concepts: list[Concept]")
    print("  - relationships: list[ConceptRelationship]")

### Deduplication Demo

The extractor deduplicates concepts that share the same name (case-insensitive),
merging keywords, source_chunk_ids, and keeping the most complete definition:

In [ ]:
from models import Difficulty

# Create duplicate concepts to demonstrate deduplication
duplicates = [
    Concept(
        id="c-1", name="Neural Network",
        definition="A computing system.",
        topics=["AI"], difficulty=Difficulty.MEDIUM,
        keywords=["neurons", "layers"],
        source_chunk_ids=["chunk-1"],
    ),
    Concept(
        id="c-2", name="neural network",
        definition="A computing system inspired by biological neural networks consisting of interconnected nodes.",
        topics=["Deep Learning"], difficulty=Difficulty.HARD,
        keywords=["deep learning", "layers"],
        source_chunk_ids=["chunk-2"],
    ),
]

deduped = extractor.deduplicate_concepts(duplicates)
print(f"Before deduplication: {len(duplicates)} concepts")
print(f"After deduplication:  {len(deduped)} concepts")
print()
for c in deduped:
    print(f"  Name: {c.name}")
    print(f"  Definition: {c.definition}")
    print(f"  Keywords: {c.keywords}")
    print(f"  Source chunks: {c.source_chunk_ids}")
    print(f"  Topics: {c.topics}")

---

## Relationship Extraction

The `RelationshipExtractor` provides comprehensive relationship analysis between concepts:
1. **LLM-powered extraction** — identifies prerequisite, related, and part_of relationships
2. **Graph construction** — builds a NetworkX directed graph from concepts and relationships
3. **Cycle validation** — detects circular prerequisites (only prerequisite edges are checked)
4. **Cycle removal** — automatically removes edges to break prerequisite cycles
5. **Visualization** — renders the relationship graph with matplotlib

In [ ]:
from models import Concept, ConceptRelationship, Difficulty
from src.extraction.relationships import RelationshipExtractor
import networkx as nx

print("RelationshipExtractor imported OK.")

### Build a Sample Concept Set & Graph

We create sample concepts and relationships to demonstrate graph operations
without requiring an LLM call:

In [ ]:
# Sample concepts
demo_concepts = [
    Concept(id="c1", name="Linear Algebra", definition="Study of vectors and matrices.",
            topics=["Math"], difficulty=Difficulty.MEDIUM),
    Concept(id="c2", name="Calculus", definition="Study of continuous change.",
            topics=["Math"], difficulty=Difficulty.MEDIUM),
    Concept(id="c3", name="Neural Networks", definition="Computing systems inspired by biology.",
            topics=["AI"], difficulty=Difficulty.HARD),
    Concept(id="c4", name="Deep Learning", definition="NNs with many hidden layers.",
            topics=["AI"], difficulty=Difficulty.HARD),
    Concept(id="c5", name="Backpropagation", definition="Gradient computation in NNs.",
            topics=["AI"], difficulty=Difficulty.HARD),
]

# Sample relationships
demo_relationships = [
    ConceptRelationship(source_concept="c1", target_concept="c3", relationship_type="prerequisite"),
    ConceptRelationship(source_concept="c2", target_concept="c3", relationship_type="prerequisite"),
    ConceptRelationship(source_concept="c3", target_concept="c4", relationship_type="prerequisite"),
    ConceptRelationship(source_concept="c5", target_concept="c3", relationship_type="part_of"),
    ConceptRelationship(source_concept="c3", target_concept="c4", relationship_type="related"),
]

# Build graph
rel_extractor = RelationshipExtractor(llm_client=LLMClient() if llm_client.available_providers else None)
# We can use the graph methods directly without LLM
graph = RelationshipExtractor.__new__(RelationshipExtractor).build_relationship_graph(
    demo_concepts, demo_relationships
)

print(f"Graph: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")
print("\nEdges:")
for u, v, data in graph.edges(data=True):
    src_name = graph.nodes[u]['name']
    tgt_name = graph.nodes[v]['name']
    print(f"  {src_name} --[{data['relationship_type']}]--> {tgt_name}")

### Cycle Validation

Validate that prerequisite edges contain no circular dependencies:

In [ ]:
extractor_instance = RelationshipExtractor.__new__(RelationshipExtractor)

is_valid, cycles = extractor_instance.validate_no_cycles(graph)
print(f"Valid (no prerequisite cycles): {is_valid}")
if cycles:
    print(f"Cycles found: {cycles}")
else:
    print("No cycles detected in prerequisite edges.")

# Now introduce a cycle for demonstration
cyclic_rels = demo_relationships + [
    ConceptRelationship(source_concept="c4", target_concept="c1", relationship_type="prerequisite"),
]
cyclic_graph = extractor_instance.build_relationship_graph(demo_concepts, cyclic_rels)
is_valid_cyclic, found_cycles = extractor_instance.validate_no_cycles(cyclic_graph)
print(f"\nWith added cycle — Valid: {is_valid_cyclic}")
print(f"Cycles: {found_cycles}")

### Cycle Removal

Automatically remove edges to break prerequisite cycles:

In [ ]:
cleaned_rels = extractor_instance.remove_cycles(cyclic_rels, demo_concepts)
print(f"Before: {len(cyclic_rels)} relationships")
print(f"After:  {len(cleaned_rels)} relationships")

# Verify cycles are gone
clean_graph = extractor_instance.build_relationship_graph(demo_concepts, cleaned_rels)
is_valid_clean, _ = extractor_instance.validate_no_cycles(clean_graph)
print(f"\nCycles removed — Valid: {is_valid_clean}")

### Graph Visualization

Render the relationship graph (red=prerequisite, blue=related, green=part_of):

In [ ]:
%matplotlib inline
extractor_instance.visualize_graph(clean_graph)

---

## Difficulty & Metadata

The `MetadataExtractor` enriches chunks and concepts with additional metadata:
1. **Difficulty classification** — assigns Easy/Medium/Hard per chunk via LLM
2. **Study time estimation** — estimates minutes to learn each concept based on difficulty + prerequisites
3. **Keyword extraction** — extracts 5-10 keywords per chunk for search/retrieval
4. **Chunk summaries** — generates 1-2 sentence summaries per chunk
5. **Document summary** — generates a 3-5 sentence overview of the full document

In [ ]:
from src.extraction.metadata import MetadataExtractor
from models import Document, DocumentMetadata, InputFormat

print("MetadataExtractor imported OK.")

### Initialize the Metadata Extractor

In [ ]:
meta_extractor = MetadataExtractor(llm_client=llm_client, batch_size=5)
print(f"MetadataExtractor ready (batch_size={meta_extractor.batch_size})")

### Difficulty Classification

Classify each chunk as Easy, Medium, or Hard using the LLM:

In [ ]:
if llm_client.available_providers:
    difficulty_map = meta_extractor.classify_difficulty(sample_chunks)
    print("Difficulty per chunk:")
    for chunk_id, diff in difficulty_map.items():
        print(f"  {chunk_id}: {diff.value}")
else:
    print("No LLM providers available — skipping difficulty classification.")
    print("The classify_difficulty() method returns a dict[str, Difficulty].")

### Study Time Estimation

Estimate study time using difficulty level and prerequisite count:
- Easy: 5-10 min base
- Medium: 15-30 min base
- Hard: 30-60 min base
- +3 min per prerequisite (capped at +15 min)

In [ ]:
# Demonstrate study time estimation with sample concepts
from models import Concept, Difficulty

demo_study_concepts = [
    Concept(id="s1", name="Variables", definition="Named storage locations.",
            topics=["Programming"], difficulty=Difficulty.EASY, prerequisites=[]),
    Concept(id="s2", name="Functions", definition="Reusable code blocks.",
            topics=["Programming"], difficulty=Difficulty.MEDIUM, prerequisites=["s1"]),
    Concept(id="s3", name="Recursion", definition="A function that calls itself.",
            topics=["Programming"], difficulty=Difficulty.HARD, prerequisites=["s1", "s2"]),
]

study_times = meta_extractor.estimate_study_time(demo_study_concepts)
print("Estimated study time per concept:")
for concept in demo_study_concepts:
    mins = study_times[concept.id]
    print(f"  {concept.name} [{concept.difficulty.value}]: {mins} min")

### Keyword Extraction

Extract 5-10 keywords per chunk for search and retrieval:

In [ ]:
if llm_client.available_providers:
    keywords_map = meta_extractor.extract_keywords(sample_chunks)
    print("Keywords per chunk:")
    for chunk_id, kws in keywords_map.items():
        print(f"  {chunk_id}: {kws}")
else:
    print("No LLM providers available — skipping keyword extraction.")
    print("The extract_keywords() method returns dict[str, list[str]].")

### Chunk Summaries

Generate concise 1-2 sentence summaries for each chunk:

In [ ]:
if llm_client.available_providers:
    chunk_summaries = meta_extractor.generate_chunk_summaries(sample_chunks)
    print("Chunk summaries:")
    for chunk_id, summary in chunk_summaries.items():
        print(f"  {chunk_id}: {summary}")
else:
    print("No LLM providers available — skipping chunk summaries.")
    print("The generate_chunk_summaries() method returns dict[str, str].")

### Document Summary

Generate a 3-5 sentence summary of the entire document:

In [ ]:
# Create a sample document
sample_doc = Document(
    content=" ".join(c.content for c in sample_chunks),
    metadata=DocumentMetadata(
        source="ml_intro.pdf",
        format=InputFormat.PDF,
        title="Introduction to Machine Learning",
    ),
)

if llm_client.available_providers:
    doc_summary = meta_extractor.generate_document_summary(sample_doc)
    print("Document summary:")
    print(f"  {doc_summary}")
else:
    print("No LLM providers available — skipping document summary.")
    print("The generate_document_summary() method returns a string.")

---

## Formula, Example, Date, People Extraction

The `ElementExtractor` class extracts structured knowledge elements from chunks:
1. **Formula extraction** — finds mathematical formulae/equations with their context
2. **Example extraction** — identifies illustrative examples with related concepts
3. **Date extraction** — extracts important dates/events with significance
4. **People extraction** — identifies key people with roles and contributions
5. **Batch processing** — processes chunks in groups of 3-5
6. **Source linking** — all results link back to source chunk IDs

In [ ]:
from src.extraction.elements import ElementExtractor
from models import Formula, Example, KeyDate, KeyPerson

print("ElementExtractor imported OK.")

### Create Physics Sample Chunks

Sample chunks containing formulae, examples, dates, and people for extraction:

In [ ]:
physics_chunks = [
    Chunk(
        id="phys-1",
        content="Albert Einstein published his theory of special relativity in 1905. "
                "The famous mass-energy equivalence formula E = mc^2 shows that "
                "energy equals mass times the speed of light squared. For example, "
                "a small amount of mass can release enormous energy in nuclear reactions.",
        document_id="doc-physics",
        chunk_index=0,
        metadata=ChunkMetadata(token_count=55, start_char=0, end_char=280),
    ),
    Chunk(
        id="phys-2",
        content="Isaac Newton formulated the law of universal gravitation in 1687. "
                "The gravitational force is given by F = G(m1*m2)/r^2. This inverse "
                "square law means doubling the distance reduces the force to one quarter. "
                "For instance, the Moon orbits Earth due to this gravitational attraction.",
        document_id="doc-physics",
        chunk_index=1,
        metadata=ChunkMetadata(token_count=60, start_char=280, end_char=580),
    ),
    Chunk(
        id="phys-3",
        content="Marie Curie won Nobel Prizes in Physics (1903) and Chemistry (1911). "
                "Her research on radioactivity led to the discovery of radium and polonium. "
                "The decay rate is described by N(t) = N0 * e^(-lambda*t), which shows "
                "exponential decrease of radioactive material over time.",
        document_id="doc-physics",
        chunk_index=2,
        metadata=ChunkMetadata(token_count=55, start_char=580, end_char=870),
    ),
]

print(f"Created {len(physics_chunks)} physics chunks for element extraction")
for c in physics_chunks:
    print(f"  [{c.id}] {c.content[:60]}...")

### Initialize the Element Extractor

In [ ]:
elem_extractor = ElementExtractor(llm_client=llm_client, batch_size=4)
print(f"ElementExtractor ready (batch_size={elem_extractor.batch_size})")

### Extract Formulae

Extract mathematical formulae/equations with their context and meaning:

In [ ]:
if llm_client.available_providers:
    formulae = elem_extractor.extract_formulae(physics_chunks)
    print(f"Extracted {len(formulae)} formulae:\n")
    for f in formulae:
        print(f"  Expression: {f.expression}")
        print(f"  Description: {f.description}")
        print(f"  Context: {f.context}")
        print(f"  Source chunk: {f.source_chunk_id}")
        print()
else:
    print("No LLM providers available — skipping formula extraction.")
    print("The extract_formulae() method returns list[Formula] with:")
    print("  - expression, description, context, source_chunk_id")

### Extract Examples

Find illustrative examples with their related concepts:

In [ ]:
if llm_client.available_providers:
    examples = elem_extractor.extract_examples(physics_chunks)
    print(f"Extracted {len(examples)} examples:\n")
    for ex in examples:
        print(f"  Title: {ex.title}")
        print(f"  Content: {ex.content[:100]}...")
        print(f"  Related concepts: {ex.related_concepts}")
        print(f"  Source chunk: {ex.source_chunk_id}")
        print()
else:
    print("No LLM providers available — skipping example extraction.")
    print("The extract_examples() method returns list[Example] with:")
    print("  - title, content, related_concepts, source_chunk_id")

### Extract Key Dates

Identify important dates and events with their significance:

In [ ]:
if llm_client.available_providers:
    dates = elem_extractor.extract_dates(physics_chunks)
    print(f"Extracted {len(dates)} key dates:\n")
    for d in dates:
        print(f"  Date: {d.date}")
        print(f"  Event: {d.event}")
        print(f"  Significance: {d.significance}")
        print(f"  Source chunk: {d.source_chunk_id}")
        print()
else:
    print("No LLM providers available — skipping date extraction.")
    print("The extract_dates() method returns list[KeyDate] with:")
    print("  - date, event, significance, source_chunk_id")

### Extract Key People

Identify important people with their roles and contributions:

In [ ]:
if llm_client.available_providers:
    people = elem_extractor.extract_people(physics_chunks)
    print(f"Extracted {len(people)} key people:\n")
    for p in people:
        print(f"  Name: {p.name}")
        print(f"  Role: {p.role}")
        print(f"  Contribution: {p.contribution}")
        print(f"  Source chunk: {p.source_chunk_id}")
        print()
else:
    print("No LLM providers available — skipping people extraction.")
    print("The extract_people() method returns list[KeyPerson] with:")
    print("  - name, role, contribution, source_chunk_id")

### Extract All Elements

The `extract_all` method runs all four extraction types in a single call:

In [ ]:
if llm_client.available_providers:
    all_elements = elem_extractor.extract_all(physics_chunks)
    print("Full element extraction results:")
    print(f"  Formulae: {len(all_elements['formulae'])}")
    print(f"  Examples: {len(all_elements['examples'])}")
    print(f"  Dates:    {len(all_elements['dates'])}")
    print(f"  People:   {len(all_elements['people'])}")
else:
    print("No LLM providers available — skipping extract_all demo.")
    print("The extract_all() method returns a dict with keys:")
    print("  'formulae', 'examples', 'dates', 'people'")

---

**Knowledge Extraction complete.** The extracted concepts, relationships, metadata, and elements
feed into downstream question generation and adaptive learning systems.